In [0]:
spark.sql("""
SELECT * ,
        upper(customer_name) AS customer_name_upper,
        date(current_timestamp()) AS processDate
FROM datamodeling.bronze.bronze_table""").createOrReplaceTempView("silver_source")

In [0]:
%sql
SELECT * FROM silver_source

order_id,order_date,customer_id,customer_name,customer_email,product_id,product_name,product_category,quantity,unit_price,payment_type,country,last_updated,customer_name_upper,processDate


## **MERGE via PySpark**

In [0]:
if spark.catalog.tableExists('datamodeling.silver.silver_table'):
    src = spark.sql(""" SELECT * FROM silver_source """)
    
    print("Merged")

else:
    spark.sql("""
              CREATE TABLE IF NOT EXISTS datamodeling.silver.silver_table
                AS
                SELECT * FROM silver_source""")

OKAY


## **MERGE via SQL**

In [0]:
%sql
CREATE TABLE IF NOT EXISTS datamodeling.silver.silver_table
AS
SELECT * FROM silver_source

num_affected_rows,num_inserted_rows


In [0]:
%sql
MERGE INTO datamodeling.silver.silver_table
USING silver_source
ON datamodeling.silver.silver_table.order_id = silver_source.order_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


In [0]:
%sql
SELECT * FROM datamodeling.silver.silver_table

order_id,order_date,customer_id,customer_name,customer_email,product_id,product_name,product_category,quantity,unit_price,payment_type,country,last_updated,customer_name_upper,processDate
1006,2022-01-06,6,David White,david.white@example.com,106,Product F,Category X,4,35.00,Apple Pay,USA,2022-01-06,DAVID WHITE,2026-05-04
1007,2022-01-07,7,Eve Black,eve.black@example.com,107,Product G,Category Z,2,40.00,PayPal,Canada,2022-01-07,EVE BLACK,2026-05-04
1001,2022-01-01,1,John Doe,john.doe@example.com,101,Product A,Category X,2,10.00,Credit Card,USA,2022-01-01,JOHN DOE,2026-05-04
1002,2022-01-02,2,Jane Smith,jane.smith@example.com,102,Product B,Category Y,1,15.00,PayPal,Canada,2022-01-02,JANE SMITH,2026-05-04
1003,2022-01-03,3,Alice Johnson,alice.johnson@example.com,103,Product C,Category X,3,20.00,Apple Pay,USA,2022-01-03,ALICE JOHNSON,2026-05-04
